In [5]:
pip install matplotlib

     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     --------- ----------------------------- 20.5/80.3 kB 65.6 kB/s eta 0:00:01
     -------------- ------------------------ 30.7/80.3 kB 93.9 kB/s eta 0:00:01
     ------------------- ------------------ 41.0/80.3 kB 131.3 kB/s eta 0:00:01
     ----------------------------- -------- 61.4/80.3 kB 173.0 kB/s eta 0:00:01
     -------------------------------------- 80.3/80.3 kB 213.5 kB/s eta 0:00:00
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/142.6 kB ? eta -:--:--
     --------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# ML Practical 1 — Bank Customer Churn Prediction using Neural Network

**Objective:** Build a neural network classifier to predict whether a bank customer will leave (churn) in the next 6 months.  
**Dataset:** Bank Customer Churn Modelling — 10,000 samples, 14 features  
**Source:** https://www.kaggle.com/barelydedicated/bank-customer-churn-modeling


---
## Step 1 — Import Libraries
We import all necessary libraries:
- `pandas` and `numpy` for data handling
- `sklearn` for preprocessing, splitting, and evaluation
- `tensorflow / keras` for building the neural network

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow version:', tf.__version__)

---
## Step 2 — Read the Dataset
We load the CSV file into a pandas DataFrame.  
`head()` shows the first 5 rows so we can visually inspect the data structure and column names.

In [ ]:
# Load dataset
# If you downloaded from Kaggle, place Churn_Modelling.csv in this folder
df = pd.read_csv('Churn_Modelling.csv')

print('Shape:', df.shape)
df.head()

---
## Step 3 — Explore the Dataset
`info()` tells us the data types of each column and whether there are any missing values.  
`value_counts()` on the target column `Exited` shows the class distribution — how many customers churned (1) vs stayed (0).

In [ ]:
df.info()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nTarget distribution:')
print(df['Exited'].value_counts())

In [ ]:
# Visualize churn distribution
df['Exited'].value_counts().plot(kind='bar', color=['steelblue','salmon'])
plt.title('Churn Distribution (0 = Stayed, 1 = Exited)')
plt.xlabel('Exited')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

---
## Step 4 — Separate Features and Target
We drop columns that are not useful for prediction:
- `RowNumber`, `CustomerId`, `Surname` — these are identifiers and carry no predictive information.

The remaining columns become the **feature matrix X**.  
The column `Exited` becomes the **target vector y** (0 = stayed, 1 = churned).

In [ ]:
# Drop identifier columns
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

print('Features shape:', X.shape)
print('Target shape  :', y.shape)
X.head()

---
## Step 5 — Encode Categorical Features
Neural networks work with numbers only. We need to convert text columns:
- `Gender` (Male/Female) → encoded with `LabelEncoder` (0/1)
- `Geography` (France/Spain/Germany) → encoded with `pd.get_dummies()` (one-hot encoding) to avoid implying an ordinal relationship

In [ ]:
# Encode Gender: Female=0, Male=1
le = LabelEncoder()
X['Gender'] = le.fit_transform(X['Gender'])

# One-hot encode Geography (drop_first avoids dummy variable trap)
X = pd.get_dummies(X, columns=['Geography'], drop_first=True)

print('Features after encoding:')
print(X.columns.tolist())
X.head()

---
## Step 6 — Split into Training and Test Sets
We split the data into 80% training and 20% testing.  
`random_state=42` ensures we get the same split every time we run the notebook.  
`stratify=y` keeps the same churn ratio in both the train and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples :', X_train.shape[0])
print('Testing  samples :', X_test.shape[0])

---
## Step 7 — Normalize the Data
Neural networks are sensitive to the scale of input features.  
`StandardScaler` transforms each feature to have **mean = 0** and **standard deviation = 1**.  

Important: We fit the scaler **only on training data** and then transform both train and test — this prevents data leakage from the test set into the model.

In [ ]:
scaler = StandardScaler()

# Fit on train, transform both train and test
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Training set after scaling (first row):')
print(X_train[0])

---
## Step 8 — Build the Neural Network
We build a **Sequential model** — layers are stacked one after another.

**Architecture:**
- **Input layer:** size equals number of features
- **Hidden Layer 1:** 64 neurons, ReLU activation — learns complex patterns
- **Dropout (0.3):** randomly drops 30% of neurons during training to prevent overfitting
- **Hidden Layer 2:** 32 neurons, ReLU activation — further refines patterns
- **Dropout (0.2):** another regularization layer
- **Output Layer:** 1 neuron, Sigmoid activation — outputs a probability between 0 and 1

**Compiler settings:**
- `adam` optimizer — adaptive learning rate, works well out of the box
- `binary_crossentropy` loss — standard for binary classification problems
- `accuracy` as the evaluation metric

In [ ]:
input_dim = X_train.shape[1]

model = Sequential([
    Dense(64, activation='relu', input_dim=input_dim),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

---
## Step 9 — Train the Model
We train the model for up to 100 epochs with a batch size of 32.  

**EarlyStopping** is a key improvement:
- It monitors validation loss during training
- If the loss stops improving for 10 consecutive epochs, training stops automatically
- `restore_best_weights=True` rolls back to the epoch where performance was best
- This prevents overfitting and saves time

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

---
## Step 10 — Plot Training History
Plotting the training and validation accuracy/loss helps us:
- Verify the model is learning (loss going down, accuracy going up)
- Detect overfitting (training accuracy much higher than validation accuracy)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'],     label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

# Loss plot
ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

---
## Step 11 — Evaluate: Accuracy Score and Confusion Matrix
We make predictions on the unseen test set.

The model outputs a probability (0 to 1). We convert it to a class label:
- Probability **>= 0.5** → predicted as **1 (Exited)**
- Probability **< 0.5** → predicted as **0 (Stayed)**

**Confusion Matrix** shows:
- **True Positives (TP):** correctly predicted as churned
- **True Negatives (TN):** correctly predicted as stayed
- **False Positives (FP):** predicted churned but actually stayed
- **False Negatives (FN):** predicted stayed but actually churned

In [ ]:
# Predict probabilities on test set
y_pred_prob = model.predict(X_test)

# Convert probabilities to binary class labels
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Accuracy Score
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy Score: {acc * 100:.2f}%')

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted: Stayed', 'Predicted: Exited'],
    yticklabels=['Actual: Stayed',    'Actual: Exited']
)
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

print('\nConfusion Matrix:')
print(cm)
print(f'  True Negatives  (Stayed correctly predicted) : {cm[0][0]}')
print(f'  False Positives (Stayed wrongly as Exited)   : {cm[0][1]}')
print(f'  False Negatives (Exited wrongly as Stayed)   : {cm[1][0]}')
print(f'  True Positives  (Exited correctly predicted) : {cm[1][1]}')

In [ ]:
# Full classification report: precision, recall, F1-score
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Stayed (0)', 'Exited (1)']))

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported required libraries |
| 2 | Loaded the dataset |
| 3 | Explored data shape, types, missing values, and class balance |
| 4 | Separated features (X) and target (y), dropped irrelevant columns |
| 5 | Encoded categorical columns (Gender, Geography) |
| 6 | Split into 80% train / 20% test with stratification |
| 7 | Normalized using StandardScaler (fit on train only) |
| 8 | Built a Sequential neural network with Dropout regularization |
| 9 | Trained with EarlyStopping to avoid overfitting |
| 10 | Plotted accuracy and loss curves |
| 11 | Evaluated with Accuracy Score and Confusion Matrix |

**Points of Improvement implemented:**
- **Dropout layers** — reduces overfitting by randomly deactivating neurons
- **EarlyStopping** — stops training when validation loss plateaus, restores best weights
- **Stratified split** — ensures both train and test sets have the same churn ratio
- **StandardScaler** — normalizes features so no single feature dominates due to scale
